Importo el dataset que contiene las respuestas a la encuesta

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

INPUT_FILE = "survey_data.xlsx"
SHEET_NAME = "Data"
OUTPUT_XLSX = "customers_clustered.xlsx"


df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
df.head()

,CID,Q109_1,Q1_1,Q1_2,Q1_3,Q1_4,Q1_5,Q1_6,Q1_7,Q1_8,...,Q2_3,Q2_4,Q2_5,Q2_6,Q2_7,Q2_8,Q2_9,Q2_10,Q2_11,Q2_12
0,1,Sample,3.0,3.0,2.0,2.0,2.0,2.0,3.0,2.0,...,4.0,5.0,4.0,4.0,5.0,5.0,5.0,4.0,4.0,5.0
1,2,Sample,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,3,non-sample,1.0,4.0,4.0,4.0,1.0,4.0,4.0,4.0,...,4.0,5.0,5.0,1.0,5.0,4.0,1.0,1.0,5.0,4.0
3,4,Sample,2.0,2.0,1.0,2.0,4.0,3.0,4.0,3.0,...,4.0,3.0,4.0,2.0,5.0,4.0,2.0,4.0,5.0,4.0
4,5,Sample,4.0,2.0,2.0,1.0,1.0,3.0,2.0,2.0,...,3.0,3.0,3.0,3.0,3.0,5.0,5.0,5.0,5.0,5.0


In [ ]:
df.shape

(1316, 25)

Armo un dataset que contenga solo a los clientes

In [ ]:
df_cust = df[df["Q109_1"] == 'non-sample']

In [ ]:
df_cust.head()

,CID,Q109_1,Q1_1,Q1_2,Q1_3,Q1_4,Q1_5,Q1_6,Q1_7,Q1_8,...,Q2_3,Q2_4,Q2_5,Q2_6,Q2_7,Q2_8,Q2_9,Q2_10,Q2_11,Q2_12
2,3,non-sample,1.0,4.0,4.0,4.0,1.0,4.0,4.0,4.0,...,4.0,5.0,5.0,1.0,5.0,4.0,1.0,1.0,5.0,4.0
124,125,non-sample,2.0,3.0,1.0,2.0,3.0,4.0,2.0,2.0,...,4.0,1.0,4.0,3.0,4.0,4.0,4.0,4.0,1.0,3.0
158,159,non-sample,3.0,4.0,1.0,1.0,1.0,2.0,3.0,2.0,...,4.0,4.0,5.0,3.0,4.0,3.0,2.0,3.0,2.0,4.0
159,160,non-sample,3.0,4.0,1.0,1.0,1.0,4.0,2.0,2.0,...,5.0,2.0,5.0,1.0,4.0,5.0,2.0,4.0,3.0,3.0
160,161,non-sample,2.0,3.0,1.0,1.0,2.0,2.0,2.0,1.0,...,3.0,3.0,4.0,2.0,4.0,4.0,4.0,1.0,3.0,4.0


In [ ]:
df_cust.shape

(513, 25)

Armo un dataset de entrada para el algoritmo de k-means, excluyo columnas CID y Q109_1

In [ ]:
excluir_cols = ["CID", "Q109_1"]

X_cust = df_cust.drop(excluir_cols, axis=1)
X_cust.head()

,Q1_1,Q1_2,Q1_3,Q1_4,Q1_5,Q1_6,Q1_7,Q1_8,Q1_9,Q1_10,...,Q2_3,Q2_4,Q2_5,Q2_6,Q2_7,Q2_8,Q2_9,Q2_10,Q2_11,Q2_12
2,1.0,4.0,4.0,4.0,1.0,4.0,4.0,4.0,4.0,4.0,...,4.0,5.0,5.0,1.0,5.0,4.0,1.0,1.0,5.0,4.0
124,2.0,3.0,1.0,2.0,3.0,4.0,2.0,2.0,2.0,4.0,...,4.0,1.0,4.0,3.0,4.0,4.0,4.0,4.0,1.0,3.0
158,3.0,4.0,1.0,1.0,1.0,2.0,3.0,2.0,2.0,2.0,...,4.0,4.0,5.0,3.0,4.0,3.0,2.0,3.0,2.0,4.0
159,3.0,4.0,1.0,1.0,1.0,4.0,2.0,2.0,3.0,2.0,...,5.0,2.0,5.0,1.0,4.0,5.0,2.0,4.0,3.0,3.0
160,2.0,3.0,1.0,1.0,2.0,2.0,2.0,1.0,2.0,4.0,...,3.0,3.0,4.0,2.0,4.0,4.0,4.0,1.0,3.0,4.0


Defino un pipeline para escalar los datos y clusterizar con k=3. Uso el algoritmo k-means y la distancia euclídea (está configurada por defecto)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])



Aplico el pipeline a X_cust para segmentar a los clientes actuales. Cada cliente, ahora, pertenecerá a uno de los 3 clusters, guardo esta información en la columna 'cluster'

In [ ]:
clusters = pipe.fit_predict(X_cust)
df_cust["cluster"] = clusters

/tmp/ipython-input-2138235070.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cust["cluster"] = clusters


In [ ]:
df_cust.head()

,CID,Q109_1,Q1_1,Q1_2,Q1_3,Q1_4,Q1_5,Q1_6,Q1_7,Q1_8,...,Q2_4,Q2_5,Q2_6,Q2_7,Q2_8,Q2_9,Q2_10,Q2_11,Q2_12,cluster
2,3,non-sample,1.0,4.0,4.0,4.0,1.0,4.0,4.0,4.0,...,5.0,5.0,1.0,5.0,4.0,1.0,1.0,5.0,4.0,1
124,125,non-sample,2.0,3.0,1.0,2.0,3.0,4.0,2.0,2.0,...,1.0,4.0,3.0,4.0,4.0,4.0,4.0,1.0,3.0,0
158,159,non-sample,3.0,4.0,1.0,1.0,1.0,2.0,3.0,2.0,...,4.0,5.0,3.0,4.0,3.0,2.0,3.0,2.0,4.0,1
159,160,non-sample,3.0,4.0,1.0,1.0,1.0,4.0,2.0,2.0,...,2.0,5.0,1.0,4.0,5.0,2.0,4.0,3.0,3.0,1
160,161,non-sample,2.0,3.0,1.0,1.0,2.0,2.0,2.0,1.0,...,3.0,4.0,2.0,4.0,4.0,4.0,1.0,3.0,4.0,2


In [ ]:
df_cust.shape

(513, 26)

Guardo la segmentación de clientes en un excel para analizar

In [ ]:
df_cust.to_excel(OUTPUT_XLSX, index=False)


